In [1]:
from datasets import load_dataset
dataset = list(load_dataset("lmarena-ai/arena-human-preference-100k")["train"])

In [2]:
print(f"Total number of samples: {len(dataset)}")
dataset_english = [d for d in dataset if d["language"] == "English"]
print(f"Number of English samples: {len(dataset_english)}")

Total number of samples: 106134
Number of English samples: 57675


In [48]:
from matplotlib import pyplot as plt
from collections import Counter
import json

with open("prompts/pairwise_pref.txt", "r") as f:
    pairwise_prompt = f.read()

dataset_en_creative = [d for d in dataset_english if d["category_tag"]["criteria_v0.1"]["creativity"] and d["turn"] == 1]
print(f"Number of creative samples: {len(dataset_en_creative)}")
# dataset_en_creative[0]

print(Counter(d['winner'] for d in dataset_en_creative))

dataset_non_ties = [d for d in dataset_en_creative if d['winner'] in ["model_a", "model_b"]]
print(f"Number of non-ties: {len(dataset_non_ties)}")

dataset_short_length = [d for d in dataset_non_ties if d['conversation_a'][-1]['num_tokens'] < 300 and d['conversation_b'][-1]['num_tokens'] < 300]
print(f"Number of short length samples: {len(dataset_short_length)}")

# plot a histogram of the conversation_a response in number of words and conversation_b response in number of words
# plt.figure(figsize=(10, 5))
# plt.hist([d['conversation_a'][-1]['num_tokens'] for d in dataset_non_ties], bins=50, alpha=0.5, label='conversation_a')
# plt.hist([d['conversation_b'][-1]['num_tokens'] for d in dataset_non_ties], bins=50, alpha=0.5, label='conversation_b')
# plt.legend()
# plt.show()

# You are given two pararaphs of writing for a given instruction.
# Your task is to determine which paragraph is overall better in terms of writing quality.

# Paragraph 1:
# [[PARAGRAPH1]]

# Paragraph 2:
# [[PARAGRAPH2]]

# You must produce your answer in the following JSON format:
# {"preference": "1|2"}

# where `preference` should be "1" if you think Paragraph 1 is better, "2" if you think Paragraph 2 is better.

final_dataset = []
for didx, d in enumerate(dataset_short_length):
    sample1 = {"id": f"test-lmarena-{len(final_dataset)}", "original_id": d['question_id'], "split": "test", "sample_type": "pairwise-lmarena", "model_a": d['model_a'], "model_b": d['model_b']}
    para1, para2 = d['conversation_a'][1]["content"], d['conversation_b'][1]["content"]
    sample1["user_instruction"] = d['conversation_a'][0]["content"]
    sample1["text_input"] = pairwise_prompt.replace("[[PARAGRAPH1]]", para1).replace("[[PARAGRAPH2]]", para2)
    sample1["paragraph1"] = para1
    sample1["paragraph2"] = para2
    sample1["reference_preference"] = "1" if d['winner'] == "model_a" else "2"
    final_dataset.append(sample1)

    sample2 = {"id": f"test-lmarena-{len(final_dataset)}", "original_id": d['question_id'], "split": "test", "sample_type": "pairwise-lmarena", "model_a": d['model_b'], "model_b": d['model_a']}
    para1, para2 = d['conversation_b'][1]["content"], d['conversation_a'][1]["content"]
    sample2["user_instruction"] = d['conversation_b'][0]["content"]
    sample2["text_input"] = pairwise_prompt.replace("[[PARAGRAPH1]]", para1).replace("[[PARAGRAPH2]]", para2)
    sample2["paragraph1"] = para1
    sample2["paragraph2"] = para2
    sample2["reference_preference"] = "2" if d['winner'] == "model_a" else "1"
    final_dataset.append(sample2)

final_dataset = final_dataset[:2000] # for now, we only use 2k samples

with open("data/lmarena_filtered_2k_pairwise_balanced.json", "w") as f:
    json.dump(final_dataset, f)

# final_dataset[100]

Number of creative samples: 15198
Counter({'model_b': 5156, 'model_a': 4852, 'tie': 2719, 'tie (bothbad)': 2471})
Number of non-ties: 10008
Number of short length samples: 2674


In [50]:
# append these samples to lamp_PRGSH_test.json
with open("data/lamp_PRGSH_test.json", "r") as f:
    lamp_PRGSH_test = json.load(f)

lamp_PRGSH_test = [d for d in lamp_PRGSH_test if d["sample_type"] != "pairwise-lmarena"]

lamp_PRGSH_test += final_dataset

with open("data/lamp_PRGSH_test.json", "w") as f:
    json.dump(lamp_PRGSH_test, f, indent=4)

print(Counter(d['sample_type'] for d in lamp_PRGSH_test))

Counter({'pairwise-lmarena': 2000, 'pairwise-gold': 1206, 'pairwise-silver': 1120, 'reward': 430, 'pairwise': 404, 'pairwise-h': 300, 'pairwise-P1': 215, 'pairwise-P2': 215, 'pairwise-P3': 209, 'pairwise-P4': 199, 'pairwise-P5': 183, 'pairwise-P6': 159, 'pairwise-P7': 138})


In [42]:
# deduplicate by id: data/lamp_PRGSH_test.json
with open("data/lamp_PRGSH_test.json", "r") as f:
    lamp_PRGSH_test = json.load(f)

already_ids = set([])
deduplicated = []
for d in lamp_PRGSH_test:
    if "input_text" in d:
        continue
    if d['id'] not in already_ids:
        deduplicated.append(d)
        already_ids.add(d['id'])

with open("data/lamp_PRGSH_test.json", "w") as f:
    json.dump(deduplicated, f, indent=4)

print(Counter(d['sample_type'] for d in deduplicated))

Counter({'pairwise-lmarena': 2000, 'pairwise-gold': 1206, 'pairwise-silver': 1120, 'reward': 430, 'pairwise': 404, 'pairwise-h': 300, 'pairwise-P1': 215, 'pairwise-P2': 215, 'pairwise-P3': 209, 'pairwise-P4': 199, 'pairwise-P5': 183, 'pairwise-P6': 159, 'pairwise-P7': 138})
